[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage2_pandas.ipynb)

# 阶段二 · Pandas 数据处理（Day 6–11）

> 《Python 数据处理 20 天学习计划》阶段二配套教材。
> **开始前：文件 → 在云端硬盘中保存一份副本。**

**本阶段目标**：能用 Pandas 读取表格数据，完成筛选、缺失值清洗、多表合并、分组统计——这是「Python 处理数据」的核心，也是科研数据处理的基本功。

**使用方法**：`Shift + Enter` 逐个运行单元格；✏️ 练习先独立写，再对照文末参考答案。

## Day 6 · 认识 Pandas：Series 与 DataFrame

- **Series**：带标签的一列数据；**DataFrame**：一张二维表（可以看成若干 Series 拼在一起）。
- Pandas 在 Colab 里已预装，惯例用 `import pandas as pd` 导入。

下面先造一张本阶段反复使用的「学生成绩表」。

In [ ]:
import pandas as pd

students = pd.DataFrame({
    "学号": [101, 102, 103, 104, 105],
    "姓名": ["小明", "小红", "小刚", "小丽", "小华"],
    "班级": ["一班", "二班", "一班", "二班", "一班"],
    "数学": [88, 95, 62, 77, 85],
    "英语": [92, 85, 70, 88, 90],
})
students

In [ ]:
# 三个最常用的“看数据”操作
print(students.head(3))     # 看前 3 行（head 默认前 5 行）
print(students.shape)       # (行数, 列数)
students.info()             # 每列的类型和缺失情况

In [ ]:
students.describe()   # 数值列的统计摘要：个数、均值、最大最小……

### 读取真实文件：CSV

实际工作中数据多在 CSV 文件里。下面先把上面的表存成 CSV，再读回来——读回来用 `pd.read_csv()`。

In [ ]:
# 存成 CSV 文件（to_csv），再读回来（read_csv）
students.to_csv("students.csv", index=False)

df = pd.read_csv("students.csv")
df.head()

### ✏️ 练习 6-1
用 `head()`、`shape`、`info()` 查看刚读回来的 `df`，回答：这张表有几行几列？「数学」列的数据类型是什么？

In [ ]:
# 练习 6-1：在这里写你的代码

## Day 7 · 数据选取与过滤

| 操作 | 写法 | 说明 |
|---|---|---|
| 选一列 | `df["数学"]` | 得到 Series |
| 选多列 | `df[["姓名","数学"]]` | 注意是双层中括号 |
| 按标签取 | `df.loc[0, "姓名"]` | 行标签 + 列名 |
| 按位置取 | `df.iloc[0, 1]` | 第 0 行第 1 列 |
| 布尔筛选 | `df[df["数学"] > 80]` | 最常用！ |
| 多条件 | `df[(条件1) & (条件2)]` | `&` 且、`|` 或，条件要加括号 |

In [ ]:
df = pd.read_csv("students.csv")

# 布尔筛选：数学大于 80 的同学
df[df["数学"] > 80]

In [ ]:
# 多条件：数学大于 80 且在一班（& 表示“且”，| 表示“或”）
df[(df["数学"] > 80) & (df["班级"] == "一班")]

In [ ]:
# 筛选后只保留需要的列；sort_values 排序
result = df[df["数学"] > 80][["姓名", "班级", "数学"]]
result.sort_values("数学", ascending=False)   # ascending=False 从高到低

### ✏️ 练习 7-1
筛出「英语成绩大于等于 90」的同学，只显示姓名和英语两列。

### ✏️ 练习 7-2
筛出「二班且数学小于 80」的同学。

In [ ]:
# 练习 7-1：在这里写你的代码

In [ ]:
# 练习 7-2：在这里写你的代码

## Day 8 · 缺失值处理

真实数据经常“缺胳膊少腿”。处理缺失值三件套：

| 操作 | 写法 | 说明 |
|---|---|---|
| 发现缺失 | `df.isnull().sum()` | 看每列缺几个 |
| 填充 | `df.fillna(值)` | 用固定值/均值等填上 |
| 删除 | `df.dropna()` | 把含缺失的行删掉 |

In [ ]:
import numpy as np

# 造一份带缺失的成绩表（None 会变成 NaN，表示缺失）
messy = pd.DataFrame({
    "姓名": ["小明", "小红", "小刚", "小丽"],
    "数学": [88, None, 62, 77],
    "英语": [92, 85, None, 88],
})
print(messy.isnull().sum())   # 每列各缺几个
messy

In [ ]:
# 方式一：用均值填充数学列的缺失
messy["数学"] = messy["数学"].fillna(messy["数学"].mean())

# 方式二：删掉仍有缺失的行
clean = messy.dropna()
clean

### ✏️ 练习 8-1
重新生成上面的 `messy` 表（把 Day 8 第一个单元格再运行一遍），这次用 **60 分**填充英语列的缺失，填充后用 `isnull().sum()` 验证英语列不再有缺失。

In [ ]:
# 练习 8-1：在这里写你的代码

## Day 9 · 数据合并：concat 与 merge

- `pd.concat([表1, 表2])`：把结构相同的表**纵向堆叠**（默认 axis=0）。
- `pd.merge(表1, 表2, on="键")`：像数据库一样**按共同的列横向合并**——Day 18 学 SQL 的 JOIN 时你会发现是同一思想。

In [ ]:
# concat：新来了一批转学生，拼到原表后面
newbies = pd.DataFrame({
    "学号": [106, 107], "姓名": ["小芳", "小军"],
    "班级": ["二班", "一班"], "数学": [90, 73], "英语": [86, 79],
})
all_students = pd.concat([df, newbies], ignore_index=True)   # ignore_index 重新编号
all_students

In [ ]:
# merge：语文成绩在另一张表里，按“学号”合并进来
chinese = pd.DataFrame({
    "学号": [101, 102, 103, 104, 105, 106, 107],
    "语文": [85, 90, 66, 82, 88, 91, 75],
})
full = pd.merge(all_students, chinese, on="学号")   # on 指定共同的“键”
full

### ✏️ 练习 9-1
在 `full` 表基础上，用「列运算」新增一列 **总分 = 数学 + 英语 + 语文**（提示：`full["总分"] = full["数学"] + full["英语"] + full["语文"]`），然后按总分从高到低排序显示。

In [ ]:
# 练习 9-1：在这里写你的代码

## Day 10 · 分组聚合 groupby

**思想：拆分 → 应用 → 合并**。按某一列把数据分成若干组，对每组分别计算，再拼成结果。

In [ ]:
# 每个班的数学、英语平均分
full.groupby("班级")[["数学", "英语"]].mean()

In [ ]:
# 每个班的人数 + 数学最高分/最低分（agg 一次算多个指标）
full.groupby("班级")["数学"].agg(["count", "max", "min", "mean"])

In [ ]:
# 选读：透视表 pivot_table——和 groupby 效果类似的另一种写法
full.pivot_table(index="班级", values=["数学", "英语"], aggfunc="mean")

### ✏️ 练习 10-1
统计每个班的**语文平均分**，并按平均分从高到低排序（提示：groupby 之后用 sort_values）。

In [ ]:
# 练习 10-1：在这里写你的代码

## Day 11 · 缓冲日：补漏 + 互动答疑 + 测验 2

**今天不新学内容，安排如下：**

1. **补漏（约 1 小时）**：把 Day 6–10 没完成的部分补齐；
2. **整理问题清单（约 15 分钟）**：翻看本 notebook 末尾的「我的问题清单」，已解决的标记掉，没解决的按主题归类；
3. **互动答疑（约 30 分钟）**：和老师 / 同学 / 辅导者约一次答疑，照着问题清单逐条问，当场把答案记进清单；
4. **完成测验 2**：打开网页版学习计划中的 <b>quiz2.html</b>（10 题，即时判分含解析，60 合格 / 80 优秀）。

> 💡 **平时怎么记录问题？** 每天学习中一遇到卡住的地方，立刻在问题清单里记一行：日期、做了什么、报错信息或困惑点。问题越具体，答疑效率越高——「groupby 之后取某一列报错 KeyError」比「groupby 不会」好问得多。

---

## 📒 我的问题清单（随手记录，缓冲日答疑前整理）

> 使用方法：学习中遇到任何卡住的地方，双击本单元格，在表格里加一行。答疑后把「状态」改为已解决并写下答案要点。

| 日期 | 遇到的问题 / 报错信息 | 状态（待问 / 已解决） | 答案要点 |
|---|---|---|---|
| 示例 | groupby 之后想取某一列却报错 KeyError | 已解决 | 结果是 Series，要用 reset_index() 还原成表 |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---

## ✅ 参考答案（先独立完成，再对照）

In [ ]:
# 练习 6-1
df_check = pd.read_csv("students.csv")
print(df_check.head())
print("行数和列数：", df_check.shape)
df_check.info()   # 可以看到「数学」列是 int64（整数）

# 练习 7-1
df = pd.read_csv("students.csv")
df[df["英语"] >= 90][["姓名", "英语"]]

In [ ]:
# 练习 7-2
df[(df["班级"] == "二班") & (df["数学"] < 80)]

# 练习 8-1
messy = pd.DataFrame({
    "姓名": ["小明", "小红", "小刚", "小丽"],
    "数学": [88, None, 62, 77],
    "英语": [92, 85, None, 88],
})
messy["英语"] = messy["英语"].fillna(60)
print(messy.isnull().sum())   # 英语列应为 0（数学列还有一个缺失是正常的）

In [ ]:
# 练习 9-1
newbies = pd.DataFrame({
    "学号": [106, 107], "姓名": ["小芳", "小军"],
    "班级": ["二班", "一班"], "数学": [90, 73], "英语": [86, 79],
})
df = pd.read_csv("students.csv")
all_students = pd.concat([df, newbies], ignore_index=True)
chinese = pd.DataFrame({
    "学号": [101, 102, 103, 104, 105, 106, 107],
    "语文": [85, 90, 66, 82, 88, 91, 75],
})
full = pd.merge(all_students, chinese, on="学号")
full["总分"] = full["数学"] + full["英语"] + full["语文"]
full.sort_values("总分", ascending=False)

In [ ]:
# 练习 10-1（接着上面的 full 表）
full.groupby("班级")["语文"].mean().sort_values(ascending=False)

---

## 🎉 阶段二完成，下一站

1. 回到网页版学习计划，完成 **测验 2（quiz2.html）**；
2. 打开 [阶段三 · 文本数据进阶](https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/stage3_text.ipynb)，进入 Day 12。